# 01. 재활용품 데이터 EDA · 품질검사 · clean/damaged 비율 제어 · YOLO processed 생성

## 0. 이번 버전에서 추가된 핵심 기능

이번 데이터에는 **멀쩡한 데이터(`clean`)**와 **파손 데이터(`damaged`)**가 함께 존재합니다.

이 노트북은 두 조건을 공정하게 비교할 수 있도록 하나의 핵심 Boolean 변수로 학습 데이터 구성을 바꿉니다.

```python
USE_DAMAGED_DATA = False
```

- `False` : 멀쩡한 Training 데이터만 사용
- `True` : 멀쩡한 데이터 일부를 파손 데이터로 **교체**해서 사용

중요한 점은 `True`일 때 파손 이미지를 단순히 추가하지 않는다는 것입니다.

예를 들어 클래스당 비교 기준이 100장이고 `DAMAGED_RATIO = 0.20`이면:

```text
False → clean 100 + damaged 0 = 총 100장
True  → clean  80 + damaged 20 = 총 100장
```

즉 **총 학습 이미지 수는 유지하고, 데이터 구성만 바꿉니다.**
이렇게 해야 "파손 데이터를 추가해서 이미지 수 자체가 늘어 성능이 좋아진 것인지"와
"파손 데이터의 다양성 때문에 좋아진 것인지"를 구분할 수 있습니다.

또한 비교를 더 공정하게 하기 위해 `True`에서 사용하는 clean 80장은
`False`의 clean 100장 중 일부입니다. 즉 두 실험에서 clean 표본 자체가 완전히 바뀌지 않습니다.

Validation은 두 조건 모두 **동일한 Validation 데이터**를 사용합니다.

# 1. 필요한 라이브러리 불러오기

이 노트북은 JSON, 이미지, 표, 그래프를 다룹니다.

- `json`: annotation JSON을 읽기 위해 사용
- `cv2`: 실제 이미지 크기 확인과 bbox 시각화에 사용
- `numpy`: 수치 계산
- `pandas`: EDA 표와 CSV 생성
- `matplotlib`: 그래프 생성
- `yaml`: YOLO용 `data.yaml` 생성

여기서는 **Ultralytics를 사용하지 않습니다.**  
1번 노트북의 역할은 모델 학습 이전까지의 데이터 준비입니다.

In [ ]:
from __future__ import annotations

import json
import math
import random
import hashlib
import re
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import yaml

from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 300)


def set_korean_font():
    """Windows/macOS/Linux에서 설치된 한글 폰트를 가능한 범위에서 자동 선택합니다."""
    candidates = [
        "Malgun Gothic",
        "AppleGothic",
        "NanumGothic",
        "Noto Sans CJK KR",
    ]
    installed = {font.name for font in fm.fontManager.ttflist}

    for name in candidates:
        if name in installed:
            plt.rcParams["font.family"] = name
            break

    plt.rcParams["axes.unicode_minus"] = False


set_korean_font()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("OpenCV:", cv2.__version__)
print("Pandas:", pd.__version__)
print("Seed:", SEED)

# 2. 경로와 실험 모드 설정

입력 원천 데이터는 기존과 같습니다.

```text
../../data/processed/
├─ Training/
│  ├─ clean/
│  └─ damaged/
│     ├─ 일부파손/
│     ├─ 상당파손/
│     └─ 완전파손/
└─ Validation/
```

하지만 최종 YOLO용 데이터는 비교 실험을 위해 서로 다른 폴더에 저장합니다.

```text
../../data/processed/model_ready/
├─ clean_only/
│  ├─ images/train
│  ├─ images/val
│  ├─ labels/train
│  ├─ labels/val
│  └─ data.yaml
│
└─ with_damaged_20pct/
   ├─ images/train
   ├─ images/val
   ├─ labels/train
   ├─ labels/val
   └─ data.yaml
```

따라서 다음처럼 두 번 실행해도 서로 덮어쓰지 않습니다.

```text
1차: USE_DAMAGED_DATA = False
2차: USE_DAMAGED_DATA = True
```

### 핵심 변수

`USE_DAMAGED_DATA`만 바꾸면 됩니다.

`DAMAGED_RATIO=0.20`은 `True`일 때 각 클래스의 학습 이미지 중 약 20%를 파손 데이터로 교체한다는 뜻입니다.

`MAX_TRAIN_IMAGES_PER_CLASS=100`은 클래스별 최대 비교 예산입니다.
실제 clean 데이터가 100장보다 적으면 그 클래스는 가능한 clean 수를 기준으로 총량을 자동 결정합니다.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

PROCESSED_DIR = (
    PROJECT_ROOT / "../../data/processed"
).resolve()

TRAINING_SOURCE_DIR = PROCESSED_DIR / "Training"
VALIDATION_SOURCE_DIR = PROCESSED_DIR / "Validation"

# ============================================================
# 가장 중요한 실험 스위치
# ============================================================
USE_DAMAGED_DATA = False

# True일 때 전체 학습 이미지 중 파손 이미지가 차지할 목표 비율
DAMAGED_RATIO = 0.20

# 클래스별 비교에 사용할 최대 이미지 수
# clean이 100장보다 적으면 실제 사용 가능한 clean 수를 기준으로 맞춥니다.
MAX_TRAIN_IMAGES_PER_CLASS = 100

# clean-only와 damaged-mix에서 같은 기준 표본을 재현하기 위한 seed
SAMPLING_SEED = 42

if not (0.0 <= DAMAGED_RATIO <= 1.0):
    raise ValueError("DAMAGED_RATIO는 0~1 사이여야 합니다.")

if MAX_TRAIN_IMAGES_PER_CLASS <= 0:
    raise ValueError("MAX_TRAIN_IMAGES_PER_CLASS는 1 이상이어야 합니다.")

DAMAGE_PERCENT = int(round(DAMAGED_RATIO * 100))

DATASET_VARIANT = (
    f"with_damaged_{DAMAGE_PERCENT}pct"
    if USE_DAMAGED_DATA
    else "clean_only"
)

MODEL_READY_ROOT = PROCESSED_DIR / "model_ready"
OUTPUT_DATASET_DIR = MODEL_READY_ROOT / DATASET_VARIANT

REPORT_ROOT = (
    PROJECT_ROOT / "../models/yolo/01_experiment_augmentation/report"
).resolve()

REPORT_DIR = REPORT_ROOT / "preprocess" / DATASET_VARIANT

for required_dir in [TRAINING_SOURCE_DIR, VALIDATION_SOURCE_DIR]:
    if not required_dir.exists():
        raise FileNotFoundError(
            "필요한 원천 폴더를 찾지 못했습니다.\n"
            f"예상 경로: {required_dir}\n"
            "PROJECT_ROOT와 ../../data/processed 경로를 확인하세요."
        )

OVERWRITE_OUTPUT_VARIANT = True

DROP_IF_TARGET_NOT_ANNOTATED = True
DROP_IF_UNKNOWN_OBJECT_EXISTS = True
KEEP_ALL_KNOWN_OBJECTS = True

print("PROJECT_ROOT             :", PROJECT_ROOT)
print("TRAINING_SOURCE_DIR      :", TRAINING_SOURCE_DIR)
print("VALIDATION_SOURCE_DIR    :", VALIDATION_SOURCE_DIR)
print("USE_DAMAGED_DATA         :", USE_DAMAGED_DATA)
print("DAMAGED_RATIO            :", DAMAGED_RATIO)
print("MAX_TRAIN_IMAGES_PER_CLASS:", MAX_TRAIN_IMAGES_PER_CLASS)
print("DATASET_VARIANT          :", DATASET_VARIANT)
print("OUTPUT_DATASET_DIR       :", OUTPUT_DATASET_DIR)
print("REPORT_DIR               :", REPORT_DIR)

# 3. 왜 폴더 클래스와 JSON 객체 클래스를 둘 다 확인해야 할까?

원본 구조는 대략 다음과 같습니다.

```text
Training/images/고철류/프라이팬/...jpg
Training/labels/고철류/프라이팬/...json
```

폴더 경로를 보면 “이 이미지는 `고철류/프라이팬` 이미지다”라고 생각할 수 있습니다.
하지만 실제 detection 정답은 **JSON 안의 `Bounding` 객체**에 들어 있습니다.

한 이미지에는 여러 객체가 있을 수 있고, 폴더명이 잘못 분류된 경우도 있을 수 있습니다.
따라서 다음 두 개를 구분합니다.

- **목표 클래스(target class)**: 이미지가 들어 있는 폴더가 의도한 클래스
- **실제 annotation class**: JSON의 각 `Bounding.CLASS + DETAILS`

이 노트북은 폴더명을 강제로 정답으로 덮어쓰지 않습니다.  
목표 클래스가 JSON에 실제로 없다면 **의심 샘플로 기록하고 기본적으로 제외**합니다.

# 4. 대분류 이름 정규화 규칙

원본 폴더와 JSON에서 같은 의미의 대분류가 조금 다르게 적힌 사례가 있습니다.

예:

- 폴더: `나무`
- JSON: `나무류`

이것을 다른 클래스로 취급하면 안 되므로 다음처럼 이름을 통일합니다.

```text
나무류       → 나무
비닐류       → 비닐
스티로폼류   → 스티로폼
유리병류     → 유리병
페트병류     → 페트병
```

이 작업은 **라벨 의미를 바꾸는 것이 아니라 문자열 표기를 통일하는 작업**입니다.

In [ ]:
CANONICAL_MAJOR = {
    "나무류": "나무",
    "비닐류": "비닐",
    "스티로폼류": "스티로폼",
    "유리병류": "유리병",
    "페트병류": "페트병",
}


def canonical_major(name: str) -> str:
    """같은 의미지만 표기가 다른 대분류 이름을 하나로 통일합니다."""
    name = str(name).strip()
    return CANONICAL_MAJOR.get(name, name)

# 5. Training / Validation 폴더를 재귀적으로 탐색

새 데이터는 폴더 깊이가 일정하지 않습니다.

예를 들어 Training에는 `clean`뿐 아니라 `damaged/일부파손`, `damaged/상당파손`,
`damaged/완전파손` 같은 중간 폴더가 존재할 수 있습니다.

따라서 특정 깊이를 가정하지 않고 `rglob()`으로 모든 이미지와 JSON을 재귀적으로 찾습니다.

이 셀에서 먼저 확인할 내용은:

- Training 이미지 수
- Training JSON 수
- Validation 이미지 수
- Validation JSON 수
- 실제 발견한 파일 경로 예시

입니다.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

source_roots = {
    "Training": TRAINING_SOURCE_DIR,
    "Validation": VALIDATION_SOURCE_DIR,
}

image_paths = []
json_paths = []

for split, root in source_roots.items():
    image_paths.extend(
        p.resolve()
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    json_paths.extend(
        p.resolve()
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() == ".json"
    )

image_paths = sorted(set(image_paths))
json_paths = sorted(set(json_paths))

# 뒤쪽 기존 분석 코드와 호환되도록 이름은 유지하되 값은 실제 파일 경로 문자열입니다.
image_members = [str(p) for p in image_paths]
json_members = [str(p) for p in json_paths]

print(f"전체 이미지    : {len(image_paths):,}")
print(f"전체 JSON 라벨 : {len(json_paths):,}")

for split, root in source_roots.items():
    n_images = sum(root in p.parents for p in image_paths)
    n_json = sum(root in p.parents for p in json_paths)
    print(f"{split:>10} | images={n_images:,} | json={n_json:,}")

print("\nJSON 경로 예시:")
for p in json_paths[:10]:
    print("-", p)

# 6. 폴더 기준 클래스 목록 만들기

우리 프로젝트에서 학습할 클래스 체계는 **Training 폴더에 실제로 존재하는 대분류/소분류 조합**을 기준으로 고정합니다.

예를 들어 `고철류/주전자`와 `도기류/주전자`는 소분류 이름이 같더라도 다른 클래스입니다.

그래서 클래스명을 단순히 `주전자`라고 만들지 않고 다음처럼 만듭니다.

```text
고철류/주전자
도기류/주전자
```

이 방식은 서로 다른 대분류의 같은 소분류 이름이 충돌하는 것을 막아줍니다.

In [ ]:
LABEL_MARKERS = {
    "Training_라벨링데이터": "Training",
    "Validation_라벨링데이터": "Validation",
}

SOURCE_MARKERS = {
    "Training_원천데이터": "Training",
    "Validation_원천데이터": "Validation",
}

def parse_folder_target(path_value):
    """
    경로의 중간 깊이가 어떻든 '*_라벨링데이터' 또는 '*_원천데이터'를 찾아
    그 다음 폴더를 대분류, 그 다음 폴더를 소분류로 읽습니다.
    """
    path = Path(path_value)
    parts = path.parts

    marker_index = None
    split = None

    for idx, part in enumerate(parts):
        if part in LABEL_MARKERS:
            marker_index = idx
            split = LABEL_MARKERS[part]
            break
        if part in SOURCE_MARKERS:
            marker_index = idx
            split = SOURCE_MARKERS[part]
            break

    if marker_index is None:
        raise ValueError(f"라벨/원천 marker를 찾지 못했습니다: {path}")

    if len(parts) <= marker_index + 2:
        raise ValueError(f"대분류/소분류를 읽기에는 경로가 너무 짧습니다: {path}")

    major = canonical_major(parts[marker_index + 1])
    detail = parts[marker_index + 2]

    return split, major, detail

# 클래스 체계는 Training JSON 폴더의 대분류/소분류 조합으로 고정합니다.
training_class_pairs = set()

for json_path in json_paths:
    split, major, detail = parse_folder_target(json_path)
    if split == "Training":
        training_class_pairs.add((major, detail))

CLASS_PAIRS = sorted(training_class_pairs)
CLASS_NAMES = [f"{major}/{detail}" for major, detail in CLASS_PAIRS]

CLASS_TO_ID = {
    class_name: class_id
    for class_id, class_name in enumerate(CLASS_NAMES)
}
ID_TO_CLASS = {
    class_id: class_name
    for class_name, class_id in CLASS_TO_ID.items()
}

print("Training 기준 최종 클래스 수:", len(CLASS_NAMES))

display(
    pd.DataFrame({
        "class_id": range(len(CLASS_NAMES)),
        "class_name": CLASS_NAMES,
    })
)

## 6-1. Training 이미지가 clean인지 damaged인지 구분하기

Training 경로에는 두 종류가 있습니다.

```text
Training/clean/...                 → clean
Training/damaged/일부파손/...       → damaged / 일부파손
Training/damaged/상당파손/...       → damaged / 상당파손
Training/damaged/완전파손/...       → damaged / 완전파손
```

Validation은 학습 데이터 교체 대상이 아니므로 별도로 `validation`으로 기록합니다.

이 정보는 나중에 클래스별로 clean과 damaged를 몇 장씩 보유하고 있는지 계산하고,
`USE_DAMAGED_DATA`에 따라 실제 사용할 표본을 고를 때 사용합니다.

In [ ]:
def get_source_condition(path_value):
    path = Path(path_value)
    parts = list(path.parts)

    if "Validation" in parts:
        return "validation", "validation"

    if "Training" not in parts:
        return "unknown", "unknown"

    if "damaged" in parts:
        damaged_index = parts.index("damaged")

        if len(parts) > damaged_index + 1:
            damage_level = parts[damaged_index + 1]
        else:
            damage_level = "unknown_damaged"

        return "damaged", damage_level

    return "clean", "clean"

# 간단한 경로 판별 예시를 출력합니다.
for example_path in json_paths[:5]:
    print(get_source_condition(example_path), example_path)

# 7. 폴더 기준 이미지 수 EDA

먼저 annotation 내용을 보기 전에 폴더만 기준으로 클래스별 이미지 수를 셉니다.

이 표는 “데이터를 어떤 의도로 수집했는가”를 보여줍니다.
반면 나중에 계산하는 객체 수는 “실제 JSON annotation에는 무엇이 들어 있는가”를 보여줍니다.

두 값이 다를 수 있다는 것이 중요합니다.

In [ ]:
folder_rows = []

for json_path in json_paths:
    split, major, detail = parse_folder_target(json_path)
    source_condition, damage_level = get_source_condition(json_path)

    folder_rows.append({
        "split": split,
        "major": major,
        "detail": detail,
        "class_name": f"{major}/{detail}",
        "source_condition": source_condition,
        "damage_level": damage_level,
        "json_member": str(json_path),
    })

folder_df = pd.DataFrame(folder_rows)

folder_support = (
    folder_df
    .groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Training", "Validation"], fill_value=0)
    .reindex(CLASS_NAMES, fill_value=0)
)

display(folder_support)

print("Training 총 JSON 수  :", int(folder_support["Training"].sum()))
print("Validation 총 JSON 수:", int(folder_support["Validation"].sum()))

In [ ]:
for split in ["Training", "Validation"]:
    counts = folder_support.loc[folder_support[split] > 0, split]

    print(f"[{split}] 클래스별 이미지 개수 분포")
    display(counts.value_counts().sort_index().rename("class_count").to_frame())

## 7-1. 클래스별 clean / damaged 보유량 확인

이 표는 이번 비교 실험에서 가장 중요합니다.

각 클래스마다:

- clean 이미지가 몇 장 있는지
- damaged 이미지가 몇 장 있는지
- damaged 내부에서 일부/상당/완전 파손이 어떻게 분포하는지

확인합니다.

`USE_DAMAGED_DATA=True`라고 해도 damaged가 부족한 클래스는 억지로 중복 복제하지 않습니다.
부족한 damaged 수만큼 clean이 자동으로 채워져 **총 학습량은 clean-only 기준과 동일하게 유지**됩니다.

In [ ]:
training_folder_df = folder_df[
    folder_df["split"].eq("Training")
].copy()

condition_support = (
    training_folder_df
    .groupby(["class_name", "source_condition"])
    .size()
    .unstack(fill_value=0)
    .reindex(CLASS_NAMES, fill_value=0)
    .reindex(columns=["clean", "damaged"], fill_value=0)
)

condition_support["total"] = (
    condition_support["clean"] + condition_support["damaged"]
)

display(condition_support)

print("clean 총 이미지 후보  :", int(condition_support["clean"].sum()))
print("damaged 총 이미지 후보:", int(condition_support["damaged"].sum()))

if (training_folder_df["source_condition"] == "damaged").any():
    damage_level_support = (
        training_folder_df[
            training_folder_df["source_condition"].eq("damaged")
        ]
        .groupby(["class_name", "damage_level"])
        .size()
        .unstack(fill_value=0)
        .reindex(CLASS_NAMES, fill_value=0)
    )

    display(damage_level_support)
else:
    print("damaged 데이터가 발견되지 않았습니다.")

## 클래스 균형 그래프 읽는 법

가로축은 클래스이고 세로축은 이미지 수입니다.

모든 막대 높이가 비슷하면 이미지 단위 클래스 균형이 비교적 좋습니다.
하지만 **이미지 수가 같다고 객체 수까지 같은 것은 아닙니다.**
한 이미지에 여러 객체가 annotation되어 있을 수 있기 때문입니다.

In [ ]:
plot_support = folder_support.reset_index()

plt.figure(figsize=(16, 6))
plt.bar(np.arange(len(plot_support)), plot_support["Training"])
plt.xlabel("Class index")
plt.ylabel("Training image count")
plt.title("Folder-based Training images per class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
plt.bar(np.arange(len(plot_support)), plot_support["Validation"])
plt.xlabel("Class index")
plt.ylabel("Validation image count")
plt.title("Folder-based Validation images per class")
plt.tight_layout()
plt.show()

# 8. JSON과 이미지 연결 규칙

원천 Training/Validation 데이터에서는 이미지와 JSON이 같은 구조를 사용하고 `images` / `labels`만 다릅니다.

```text
Training/images/고철류/프라이팬/.../sample.jpg
Training/labels/고철류/프라이팬/.../sample.json
```

아래 함수는 JSON 경로를 이미지 경로로 바꿉니다.

In [ ]:
def json_to_image_path(json_path_value):
    """
    JSON 경로의 라벨링데이터 marker를 같은 branch의 원천데이터 marker로 바꿔
    대응 이미지 파일을 찾습니다.

    예:
    Training/.../Training_라벨링데이터/고철류/고철/a.Json
      -> Training/.../Training_원천데이터/고철류/고철/a.jpg
    """
    json_path = Path(json_path_value)
    parts = list(json_path.parts)

    replacements = {
        "Training_라벨링데이터": "Training_원천데이터",
        "Validation_라벨링데이터": "Validation_원천데이터",
    }

    replaced = False
    for idx, part in enumerate(parts):
        if part in replacements:
            parts[idx] = replacements[part]
            replaced = True
            break

    if not replaced:
        raise ValueError(f"라벨링데이터 marker가 없는 JSON 경로입니다: {json_path}")

    base = Path(*parts).with_suffix("")

    # Linux(Kaggle)에서는 확장자 대소문자가 구분되므로 가능한 확장자를 모두 검사합니다.
    for ext in sorted(IMAGE_EXTENSIONS):
        for candidate_ext in [ext, ext.upper()]:
            candidate = base.with_suffix(candidate_ext)
            if candidate.exists():
                return candidate.resolve()

    # 발견되지 않았을 때 일반적인 예상 경로를 반환해 audit에서 missing 처리합니다.
    return base.with_suffix(".jpg").resolve()

# 9. BOX와 POLYGON을 Detection bbox로 읽는 함수

YOLO Detection은 직사각형 bounding box가 필요합니다.

## BOX

원본에 `x1, y1, x2, y2`가 있으면 그대로 사용할 수 있습니다.

## POLYGON

일부 annotation은 객체 외곽을 여러 점으로 그린 polygon입니다.
현재 프로젝트는 **segmentation이 아니라 object detection**을 학습하므로 polygon의 모든 점을 포함하는 가장 작은 축 정렬 직사각형을 만듭니다.

즉,

```text
polygon의 최소 x → x1
polygon의 최소 y → y1
polygon의 최대 x → x2
polygon의 최대 y → y2
```

이렇게 하면 polygon 정보 일부는 단순화되지만 Detection bbox로 일관되게 학습할 수 있습니다.

In [ ]:
def polygon_points_to_xyxy(polygon_points):
    points = []

    for item in polygon_points or []:
        if not item:
            continue

        raw_value = next(iter(item.values()))
        x, y = str(raw_value).split(",")
        points.append((float(x), float(y)))

    if not points:
        raise ValueError("PolygonPoint가 비어 있습니다.")

    xs = [point[0] for point in points]
    ys = [point[1] for point in points]

    return min(xs), min(ys), max(xs), max(ys)


def bounding_to_xyxy(bounding: dict):
    """Bounding 하나를 x1,y1,x2,y2로 변환하고 원본 geometry 종류도 반환합니다."""
    if all(key in bounding for key in ("x1", "y1", "x2", "y2")):
        return (
            float(bounding["x1"]),
            float(bounding["y1"]),
            float(bounding["x2"]),
            float(bounding["y2"]),
            "BOX",
        )

    if bounding.get("PolygonPoint"):
        x1, y1, x2, y2 = polygon_points_to_xyxy(bounding["PolygonPoint"])
        return x1, y1, x2, y2, "POLYGON->BOX"

    raise ValueError("BOX 좌표도 PolygonPoint도 없는 annotation입니다.")

# 10. 이미지 실제 해상도와 JSON RESOLUTION 읽기

bbox 좌표 정규화에는 이미지의 가로/세로 크기가 꼭 필요합니다.

원본 JSON에는 `RESOLUTION`도 있지만, 메타데이터는 잘못될 수 있습니다.
따라서 이 노트북은 **OpenCV로 실제 이미지를 디코딩해서 얻은 width/height를 정답으로 사용**합니다.

JSON 해상도는 품질검사 참고용으로만 비교합니다.

In [ ]:
def parse_resolution(value):
    """JSON RESOLUTION 문자열을 (width, height)로 바꿉니다."""
    match = re.match(r"\s*(\d+)\s*[xX*]\s*(\d+)\s*", str(value))

    if not match:
        return None, None

    return int(match.group(1)), int(match.group(2))

def load_image(image_path_value):
    """실제 파일 시스템의 이미지를 OpenCV로 읽습니다."""
    return cv2.imread(str(image_path_value), cv2.IMREAD_COLOR)

# 11. 원본 전체 품질검사 + EDA 테이블 만들기

이 셀이 원천 Training/Validation 데이터을 가장 꼼꼼하게 읽는 핵심 단계입니다.

각 이미지마다 다음을 기록합니다.

- split
- 폴더 목표 클래스
- 실제 이미지 width / height
- JSON RESOLUTION
- 실내/실외
- 주간/야간
- 객체 수
- JSON에 목표 클래스가 존재하는지
- Training에서 정의한 학습 클래스 체계 밖의 객체가 존재하는지
- 이미지 SHA-1 hash

각 객체마다 다음을 기록합니다.

- JSON의 실제 클래스
- BOX인지 POLYGON인지
- bbox 좌표
- bbox 면적이 전체 이미지에서 차지하는 비율
- bbox 가로/세로 비율
- 폴더 목표 클래스와 같은 객체인지

In [ ]:
image_rows = []
object_rows = []
issue_rows = []

known_class_pairs = set(CLASS_PAIRS)

for json_member in sorted(json_members):
    json_path = Path(json_member)
    split, folder_major, folder_detail = parse_folder_target(json_path)

    target_pair = (folder_major, folder_detail)
    target_class_name = f"{folder_major}/{folder_detail}"
    source_condition, damage_level = get_source_condition(json_path)

    try:
        try:
            with open(json_path, "r", encoding="utf-8-sig") as file:
                payload = json.load(file)
        except UnicodeDecodeError:
            with open(json_path, "r", encoding="cp949") as file:
                payload = json.load(file)
    except Exception as error:
        issue_rows.append({
            "issue_type": "json_read_failed",
            "split": split,
            "json_member": str(json_path),
            "image_member": None,
            "detail": repr(error),
        })
        continue

    image_path = json_to_image_path(json_path)
    image_member = str(image_path)

    if not image_path.exists():
        issue_rows.append({
            "issue_type": "missing_image",
            "split": split,
            "json_member": str(json_path),
            "image_member": image_member,
            "detail": "JSON과 같은 stem의 원천 이미지를 찾지 못했습니다.",
        })
        continue

    raw_image = image_path.read_bytes()
    sha1 = hashlib.sha1(raw_image).hexdigest()
    image = load_image(image_path)

    if image is None:
        issue_rows.append({
            "issue_type": "decode_failed",
            "split": split,
            "json_member": str(json_path),
            "image_member": image_member,
            "detail": "OpenCV 이미지 decode 실패",
        })
        continue

    height, width = image.shape[:2]
    meta_width, meta_height = parse_resolution(payload.get("RESOLUTION", ""))

    if (meta_width, meta_height) != (width, height):
        issue_rows.append({
            "issue_type": "resolution_metadata_mismatch",
            "split": split,
            "json_member": str(json_path),
            "image_member": image_member,
            "detail": f"actual={width}x{height}, metadata={meta_width}x{meta_height}",
        })

    bounds = payload.get("Bounding") or []
    actual_pairs = []
    unknown_pairs = []
    valid_object_count = 0

    for object_index, bounding in enumerate(bounds):
        major = canonical_major(bounding.get("CLASS", "UNKNOWN"))
        detail = str(bounding.get("DETAILS", "UNKNOWN")).strip()
        object_pair = (major, detail)
        class_name = f"{major}/{detail}"

        actual_pairs.append(object_pair)

        if object_pair not in known_class_pairs:
            unknown_pairs.append(object_pair)

        try:
            x1, y1, x2, y2, geometry = bounding_to_xyxy(bounding)

            clipped_x1 = float(np.clip(x1, 0, width))
            clipped_x2 = float(np.clip(x2, 0, width))
            clipped_y1 = float(np.clip(y1, 0, height))
            clipped_y2 = float(np.clip(y2, 0, height))

            if clipped_x2 <= clipped_x1 or clipped_y2 <= clipped_y1:
                raise ValueError("clip 후 bbox width/height가 0 이하")

            bbox_width = clipped_x2 - clipped_x1
            bbox_height = clipped_y2 - clipped_y1
            bbox_area_ratio = (bbox_width * bbox_height) / (width * height)
            bbox_aspect_ratio = bbox_width / bbox_height

            valid_object_count += 1

            object_rows.append({
                "split": split,
                "image_member": image_member,
                "json_member": str(json_path),
                "object_index": object_index,
                "folder_target_class": target_class_name,
                "source_condition": source_condition,
                "damage_level": damage_level,
                "object_major": major,
                "object_detail": detail,
                "object_class": class_name,
                "is_target_object": object_pair == target_pair,
                "is_known_training_class": object_pair in known_class_pairs,
                "source_geometry": geometry,
                "x1": clipped_x1,
                "y1": clipped_y1,
                "x2": clipped_x2,
                "y2": clipped_y2,
                "bbox_width": bbox_width,
                "bbox_height": bbox_height,
                "bbox_area_ratio": bbox_area_ratio,
                "bbox_aspect_ratio": bbox_aspect_ratio,
            })

        except Exception as error:
            issue_rows.append({
                "issue_type": "invalid_annotation",
                "split": split,
                "json_member": str(json_path),
                "image_member": image_member,
                "detail": f"object_index={object_index}, error={error!r}",
            })

    target_is_annotated = target_pair in actual_pairs

    if not target_is_annotated:
        issue_rows.append({
            "issue_type": "folder_target_not_in_json",
            "split": split,
            "json_member": str(json_path),
            "image_member": image_member,
            "detail": (
                f"folder target={target_class_name}, "
                f"json objects={[f'{m}/{d}' for m, d in actual_pairs]}"
            ),
        })

    if unknown_pairs:
        issue_rows.append({
            "issue_type": "unknown_object_class",
            "split": split,
            "json_member": str(json_path),
            "image_member": image_member,
            "detail": f"unknown objects={[f'{m}/{d}' for m, d in unknown_pairs]}",
        })

    image_rows.append({
        "split": split,
        "image_member": image_member,
        "json_member": str(json_path),
        "image_name": image_path.name,
        "folder_major": folder_major,
        "folder_detail": folder_detail,
        "folder_target_class": target_class_name,
        "source_condition": source_condition,
        "damage_level": damage_level,
        "width": width,
        "height": height,
        "meta_width": meta_width,
        "meta_height": meta_height,
        "resolution_match": (meta_width, meta_height) == (width, height),
        "place": payload.get("PLACE"),
        "day_night": payload.get("DAY/NIGHT"),
        "bounding_count_declared": int(payload.get("BoundingCount", len(bounds))),
        "bounding_count_json": len(bounds),
        "valid_object_count": valid_object_count,
        "target_is_annotated": target_is_annotated,
        "unknown_object_count": len(unknown_pairs),
        "sha1": sha1,
    })

image_df = pd.DataFrame(image_rows)
object_df = pd.DataFrame(object_rows)
issue_df = pd.DataFrame(issue_rows)

print("image_df rows :", len(image_df))
print("object_df rows:", len(object_df))
print("issue_df rows :", len(issue_df))

# 12. 원본 데이터 전체 요약

여기서는 가장 기본적인 규모를 확인합니다.

`objects`는 “이미지 폴더 수”가 아니라 JSON 안에서 읽은 실제 객체 annotation 수입니다.

In [ ]:
raw_summary = (
    image_df
    .groupby("split")
    .agg(
        images=("image_member", "count"),
        valid_objects=("valid_object_count", "sum"),
        target_mismatch_images=("target_is_annotated", lambda s: int((~s).sum())),
        resolution_metadata_mismatches=("resolution_match", lambda s: int((~s).sum())),
    )
    .reindex(["Training", "Validation"])
)

display(raw_summary)

geometry_summary = object_df["source_geometry"].value_counts().rename("objects").to_frame()
display(geometry_summary)

# 13. 데이터 품질 이슈 목록

`issue_type`별 의미:

- `folder_target_not_in_json`: 폴더가 의도한 클래스가 JSON 객체에 없음
- `unknown_object_class`: Training에서 정의한 클래스 체계 밖의 객체가 annotation됨
- `resolution_metadata_mismatch`: JSON의 해상도 문자열과 실제 이미지 크기가 다름
- `invalid_annotation`: bbox 좌표가 잘못되었거나 해석할 수 없음
- `missing_image`: JSON은 있는데 대응 이미지가 없음
- `decode_failed`: 이미지 파일이 손상되어 OpenCV가 읽지 못함

**해상도 metadata mismatch는 곧바로 이미지 삭제 사유가 아닙니다.**  
실제 이미지를 정상적으로 읽을 수 있다면 실제 width/height를 사용해 안전하게 변환할 수 있습니다.

In [ ]:
if len(issue_df):
    display(issue_df["issue_type"].value_counts().rename("count").to_frame())
    display(issue_df.sort_values(["issue_type", "split"]))
else:
    print("발견된 품질 이슈가 없습니다.")

# 14. 폴더-JSON 불일치 샘플만 따로 확인

이 항목은 특히 중요합니다.

예를 들어 폴더는 `도기류/기타`인데 JSON 안에는 `도기류/그릇류`와 `전자제품/기타`만 있다면,
폴더 이름을 근거로 `도기류/기타` bbox를 만들어서는 안 됩니다.
실제로 없는 정답을 조작하는 것이 되기 때문입니다.

따라서 기본 정책은 **해당 이미지를 processed에서 제외**하는 것입니다.

In [ ]:
mismatch_df = issue_df[
    issue_df["issue_type"].eq("folder_target_not_in_json")
].copy() if len(issue_df) else pd.DataFrame()

print("폴더 목표 클래스가 JSON에 없는 이미지 수:", len(mismatch_df))

if len(mismatch_df):
    display(mismatch_df[["split", "image_member", "detail"]])

# 15. 촬영 환경 EDA: 주간/야간, 실내/실외

모델은 학습 데이터에서 본 환경에 강하고, 보지 못한 환경에는 약할 수 있습니다.

예를 들어 학습 데이터가 모두 주간이라면 실제 서비스에서 밤에 촬영한 사진에 대한 성능은 별도로 검증해야 합니다.
이미지 증강으로 어둡게 만드는 것은 도움이 될 수 있지만 **실제 야간 데이터 자체를 완전히 대체하지는 못합니다.**

In [ ]:
place_table = (
    image_df.groupby(["split", "place"])
    .size()
    .rename("images")
    .reset_index()
)

day_table = (
    image_df.groupby(["split", "day_night"])
    .size()
    .rename("images")
    .reset_index()
)

display(place_table)
display(day_table)

In [ ]:
place_pivot = (
    place_table
    .pivot(index="place", columns="split", values="images")
    .fillna(0)
)

place_pivot.plot(kind="bar", figsize=(8, 5))
plt.ylabel("Images")
plt.title("Indoor / Outdoor Distribution")
plt.tight_layout()
plt.show()

# 16. 이미지 해상도 EDA

YOLO는 학습 중 `imgsz` 크기로 입력을 조정하기 때문에 원본을 미리 모두 640×640으로 저장할 필요가 없습니다.

오히려 지금 단계에서 JPG를 다시 리사이즈·저장하면:

- 한 번 더 압축되어 화질이 손실될 수 있고
- 원본 비율과 정보를 불필요하게 바꿀 수 있으며
- 이후 다른 `imgsz` 실험을 하기 어려워집니다.

따라서 processed에는 **원본 이미지 bytes를 그대로 복사**합니다.

In [ ]:
resolution_df = (
    image_df.groupby(["width", "height"])
    .size()
    .rename("images")
    .reset_index()
    .sort_values("images", ascending=False)
)

display(resolution_df.head(30))

plt.figure(figsize=(10, 6))
plt.scatter(image_df["width"], image_df["height"], alpha=0.35)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Original Image Resolution Distribution")
plt.tight_layout()
plt.show()

# 17. 이미지당 객체 수 EDA

폴더가 클래스별로 이미지를 수집했다고 해도 한 이미지에 다른 객체가 함께 들어 있을 수 있습니다.

Detection에서는 이미지 안에 존재하는 **학습 대상 객체를 가능한 한 모두 annotation하는 것이 중요**합니다.
그렇지 않으면 실제 객체를 background라고 잘못 가르칠 수 있기 때문입니다.

따라서 폴더 목표 클래스가 정상적으로 존재하는 이미지에서는 기본적으로 JSON에 있는 **86개 학습 대상 클래스의 모든 객체를 유지**합니다.

In [ ]:
objects_per_image = image_df["valid_object_count"].value_counts().sort_index()

display(objects_per_image.rename("images").to_frame())

plt.figure(figsize=(8, 5))
plt.bar(objects_per_image.index.astype(str), objects_per_image.values)
plt.xlabel("Objects in one image")
plt.ylabel("Number of images")
plt.title("Objects per Image")
plt.tight_layout()
plt.show()

# 18. bbox 크기 EDA

`bbox_area_ratio`는 객체 bbox 면적을 전체 이미지 면적으로 나눈 값입니다.

예:

- 0.01 → 이미지의 약 1% 면적
- 0.25 → 이미지의 약 25% 면적

작은 객체가 많으면 `imgsz`, scale 증강, mosaic 등에 대한 반응이 달라질 수 있습니다.
이 분포는 두 번째 증강 실험을 해석할 때도 중요한 참고 자료입니다.

In [ ]:
display(
    object_df["bbox_area_ratio"]
    .describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    .to_frame()
)

plt.figure(figsize=(10, 5))
plt.hist(object_df["bbox_area_ratio"], bins=40)
plt.xlabel("bbox area / image area")
plt.ylabel("Objects")
plt.title("Bounding Box Area Ratio")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log10(object_df["bbox_aspect_ratio"].clip(lower=1e-6)), bins=40)
plt.xlabel("log10(bbox width / bbox height)")
plt.ylabel("Objects")
plt.title("Bounding Box Aspect Ratio")
plt.tight_layout()
plt.show()

# 19. 실제 객체 클래스 분포 확인

이 표는 폴더가 아니라 **JSON의 실제 객체 annotation**을 기준으로 셉니다.

한 이미지에 다른 학습 대상 객체가 추가로 들어 있으면 해당 클래스 객체 수가 늘어날 수 있습니다.
따라서 폴더 기준 이미지 균형과 객체 기준 균형은 서로 다를 수 있습니다.

In [ ]:
object_support = (
    object_df[object_df["is_known_training_class"]]
    .groupby(["object_class", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Training", "Validation"], fill_value=0)
)

object_support["total"] = object_support["Training"] + object_support["Validation"]
object_support = object_support.sort_index()

display(object_support)

# 20. Train-Val 이미지 누수 검사

Validation은 모델이 학습하지 않은 데이터에서 성능을 보는 역할을 합니다.
그런데 Training과 Validation에 같은 이미지가 들어 있으면 모델이 이미 본 사진을 다시 평가하게 되어 성능이 실제보다 높아질 수 있습니다.

파일명이 다르더라도 이미지 bytes가 같을 수 있으므로 SHA-1 hash를 기준으로 검사합니다.

In [ ]:
duplicate_groups = []
cross_split_duplicate_groups = []

for sha1, group in image_df.groupby("sha1"):
    if len(group) > 1:
        duplicate_groups.append(group[["split", "image_member", "sha1"]])

        if group["split"].nunique() > 1:
            cross_split_duplicate_groups.append(group[["split", "image_member", "sha1"]])

print("전체 exact duplicate 그룹:", len(duplicate_groups))
print("Train-Val exact duplicate 그룹:", len(cross_split_duplicate_groups))

if cross_split_duplicate_groups:
    display(pd.concat(cross_split_duplicate_groups, ignore_index=True))
else:
    print("Train-Val exact duplicate 없음: PASSED")

# 21. 품질검사 통과 여부 + clean/damaged 비교용 표본 선택

이 단계는 두 부분으로 나뉩니다.

## A. 먼저 품질검사

아래와 같은 이미지는 clean/damaged 여부와 상관없이 제외합니다.

1. 폴더 목표 클래스가 JSON에 실제로 없음
2. 학습 대상 클래스 체계 밖의 unknown 객체가 존재함

이 결과를 `eligible_for_sampling`이라고 부릅니다.

## B. 그 다음 비교용 sampling

### `USE_DAMAGED_DATA=False`

클래스마다 품질검사를 통과한 **clean 데이터만** 사용합니다.
최대 `MAX_TRAIN_IMAGES_PER_CLASS`장까지 고릅니다.

### `USE_DAMAGED_DATA=True`

clean-only에서 사용했을 총 장수를 그대로 유지한 채,
그중 `DAMAGED_RATIO`만큼을 damaged로 **교체**합니다.

예를 들어 clean 후보가 충분하면:

```text
clean-only      : clean 100 + damaged  0 = 100
with-damaged 20%: clean  80 + damaged 20 = 100
```

damaged 후보가 20장보다 적다면 사용 가능한 damaged만 쓰고 나머지는 clean으로 채웁니다.
따라서 **파손 데이터 부족 때문에 전체 학습량까지 줄어들지는 않습니다.**

### 중요한 공정성 장치

clean-only에서 선택한 clean 100장을 먼저 seed로 고정합니다.
with-damaged에서는 그 100장 중 일부를 그대로 유지하고 나머지만 damaged로 교체합니다.

즉 두 실험 사이에서 clean 표본까지 전부 랜덤하게 바뀌는 문제를 막습니다.

Validation은 두 모드에서 동일하게 유지합니다.

In [ ]:
def decide_quality_eligibility(row):
    reasons = []

    if DROP_IF_TARGET_NOT_ANNOTATED and not bool(row["target_is_annotated"]):
        reasons.append("folder_target_not_in_json")

    if DROP_IF_UNKNOWN_OBJECT_EXISTS and int(row["unknown_object_count"]) > 0:
        reasons.append("unknown_object_exists")

    return len(reasons) == 0, ";".join(reasons)

quality_decisions = image_df.apply(
    decide_quality_eligibility,
    axis=1,
    result_type="expand",
)

image_df["eligible_for_sampling"] = quality_decisions[0]
image_df["quality_exclude_reason"] = quality_decisions[1]


def stable_rank(path_value, seed=SAMPLING_SEED):
    """OS/실행환경이 달라도 같은 파일은 같은 순서가 되도록 deterministic hash를 만듭니다."""
    text = f"{seed}|{path_value}"
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def pick_damaged_diverse(group, n):
    """
    damaged 표본을 가능한 한 일부/상당/완전파손에서 고르게 뽑습니다.

    각 damage_level 내부 순서는 stable hash로 고정하고,
    level들을 round-robin으로 순회하며 한 장씩 선택합니다.
    """
    if n <= 0 or len(group) == 0:
        return []

    work = group.copy()
    work["_rank"] = work["json_member"].map(stable_rank)

    level_names = sorted(work["damage_level"].dropna().astype(str).unique())
    queues = {
        level: work[work["damage_level"].astype(str).eq(level)]
        .sort_values("_rank")["json_member"].tolist()
        for level in level_names
    }

    selected = []

    while len(selected) < n:
        progressed = False

        for level in level_names:
            if queues[level] and len(selected) < n:
                selected.append(queues[level].pop(0))
                progressed = True

        if not progressed:
            break

    return selected


# 최종 선택 여부 초기화
image_df["selected_for_processed"] = False
image_df["selection_role"] = "not_selected"

sampling_plan_rows = []
selected_json_members = set()

# ------------------------------
# Training: 클래스별 controlled sampling
# ------------------------------
eligible_train = image_df[
    image_df["split"].eq("Training")
    & image_df["eligible_for_sampling"]
].copy()

for class_name in CLASS_NAMES:
    class_df = eligible_train[
        eligible_train["folder_target_class"].eq(class_name)
    ].copy()

    clean_df = class_df[
        class_df["source_condition"].eq("clean")
    ].copy()

    damaged_df = class_df[
        class_df["source_condition"].eq("damaged")
    ].copy()

    clean_df["_rank"] = clean_df["json_member"].map(stable_rank)
    clean_df = clean_df.sort_values("_rank")

    clean_available = len(clean_df)
    damaged_available = len(damaged_df)

    # 두 실험의 공통 총량 기준은 clean-only에서 실제로 확보 가능한 수입니다.
    comparison_budget = min(
        MAX_TRAIN_IMAGES_PER_CLASS,
        clean_available,
    )

    # clean-only baseline 표본을 먼저 고정합니다.
    baseline_clean_members = clean_df.head(comparison_budget)[
        "json_member"
    ].tolist()

    if USE_DAMAGED_DATA:
        desired_damaged = int(
            np.floor(comparison_budget * DAMAGED_RATIO + 0.5)
        )

        actual_damaged = min(desired_damaged, damaged_available)
        actual_clean = comparison_budget - actual_damaged

        selected_clean = baseline_clean_members[:actual_clean]
        selected_damaged = pick_damaged_diverse(
            damaged_df,
            actual_damaged,
        )
    else:
        desired_damaged = 0
        actual_damaged = 0
        actual_clean = comparison_budget
        selected_clean = baseline_clean_members
        selected_damaged = []

    selected_json_members.update(selected_clean)
    selected_json_members.update(selected_damaged)

    sampling_plan_rows.append({
        "class_name": class_name,
        "clean_available": clean_available,
        "damaged_available": damaged_available,
        "comparison_budget": comparison_budget,
        "desired_damaged": desired_damaged,
        "selected_clean": len(selected_clean),
        "selected_damaged": len(selected_damaged),
        "selected_total": len(selected_clean) + len(selected_damaged),
        "actual_damaged_ratio": (
            len(selected_damaged) / comparison_budget
            if comparison_budget > 0
            else 0.0
        ),
        "damaged_shortage": max(0, desired_damaged - len(selected_damaged)),
    })

    image_df.loc[
        image_df["json_member"].isin(selected_clean),
        "selection_role",
    ] = "selected_clean"

    image_df.loc[
        image_df["json_member"].isin(selected_damaged),
        "selection_role",
    ] = "selected_damaged"

# ------------------------------
# Validation: 두 모드에서 동일하게 전부 유지
# ------------------------------
eligible_val_members = image_df[
    image_df["split"].eq("Validation")
    & image_df["eligible_for_sampling"]
]["json_member"].tolist()

selected_json_members.update(eligible_val_members)

image_df.loc[
    image_df["json_member"].isin(eligible_val_members),
    "selection_role",
] = "selected_validation"

image_df["selected_for_processed"] = image_df["json_member"].isin(
    selected_json_members
)

# 뒤쪽 기존 전처리 코드와 호환되도록 keep_for_processed 이름도 유지합니다.
image_df["keep_for_processed"] = image_df["selected_for_processed"]

# 선택되지 않은 이유 정리
image_df["exclude_reason"] = image_df["quality_exclude_reason"].copy()

not_selected_mask = (
    image_df["eligible_for_sampling"]
    & ~image_df["selected_for_processed"]
)

image_df.loc[
    not_selected_mask & image_df["split"].eq("Training"),
    "exclude_reason",
] = "not_selected_by_controlled_sampling"

sampling_plan_df = pd.DataFrame(sampling_plan_rows)

# 공정성 핵심 검증: 선택 총량은 항상 clean-only comparison budget과 같아야 합니다.
if not (sampling_plan_df["selected_total"] == sampling_plan_df["comparison_budget"]).all():
    bad = sampling_plan_df[
        sampling_plan_df["selected_total"] != sampling_plan_df["comparison_budget"]
    ]
    display(bad)
    raise ValueError("일부 클래스에서 clean-only와 비교 총량이 달라졌습니다.")

if not USE_DAMAGED_DATA and int(sampling_plan_df["selected_damaged"].sum()) != 0:
    raise ValueError("USE_DAMAGED_DATA=False인데 damaged 이미지가 선택되었습니다.")

print("DATASET_VARIANT:", DATASET_VARIANT)
print("USE_DAMAGED_DATA:", USE_DAMAGED_DATA)

display(sampling_plan_df)

print("Training 선택 구성")
display(
    image_df[
        image_df["split"].eq("Training")
        & image_df["selected_for_processed"]
    ]
    .groupby(["source_condition", "damage_level"])
    .size()
    .rename("images")
    .reset_index()
)

print("클래스별 실제 damaged 비율 요약")
display(
    sampling_plan_df[
        [
            "class_name",
            "comparison_budget",
            "selected_clean",
            "selected_damaged",
            "selected_total",
            "actual_damaged_ratio",
            "damaged_shortage",
        ]
    ]
)

excluded_df = image_df[~image_df["selected_for_processed"]].copy()

# 22. YOLO Detection 라벨 형식 이해하기

YOLO txt 한 줄은 다음 형식입니다.

```text
class_id x_center y_center width height
```

여기서 좌표는 픽셀이 아니라 **0~1 사이의 비율**입니다.

예를 들어 이미지 width가 1000px이고 bbox width가 200px이면:

```text
normalized width = 200 / 1000 = 0.2
```

정규화하는 이유는 이미지 해상도가 달라도 같은 방식으로 모델이 좌표를 다룰 수 있기 때문입니다.

In [ ]:
def xyxy_to_yolo(x1, y1, x2, y2, image_width, image_height):
    """pixel xyxy bbox를 YOLO normalized xc,yc,w,h로 변환합니다."""
    x_center = ((x1 + x2) / 2.0) / image_width
    y_center = ((y1 + y2) / 2.0) / image_height
    box_width = (x2 - x1) / image_width
    box_height = (y2 - y1) / image_height

    return x_center, y_center, box_width, box_height

# 23. 모드별 YOLO dataset 폴더 생성

이번에는 clean-only와 damaged-mix 결과를 서로 덮어쓰지 않습니다.

```text
../../data/processed/model_ready/
├─ clean_only/
└─ with_damaged_20pct/
```

현재 실행에서 실제 사용하는 출력 폴더는 `OUTPUT_DATASET_DIR`입니다.

`OVERWRITE_OUTPUT_VARIANT=True`이면 **현재 variant 폴더만** 지웠다가 다시 만듭니다.
다른 variant와 원천 `Training/`, `Validation/`은 절대로 삭제하지 않습니다.

In [ ]:
if OUTPUT_DATASET_DIR.exists() and OVERWRITE_OUTPUT_VARIANT:
    shutil.rmtree(OUTPUT_DATASET_DIR)

if REPORT_DIR.exists() and OVERWRITE_OUTPUT_VARIANT:
    shutil.rmtree(REPORT_DIR)

for split in ["train", "val"]:
    (OUTPUT_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Training 원천 보존   :", TRAINING_SOURCE_DIR.exists())
print("Validation 원천 보존 :", VALIDATION_SOURCE_DIR.exists())
print("현재 dataset variant :", DATASET_VARIANT)
print("YOLO output          :", OUTPUT_DATASET_DIR)
print("전처리 report         :", REPORT_DIR)

# 24. 파일명 충돌 방지

현재 데이터에서는 이미지 basename이 겹치지 않지만, 앞으로 다른 데이터가 추가되면 같은 `sample.jpg` 이름이 여러 폴더에 있을 수 있습니다.

processed를 한 폴더로 평평하게(flat) 모으기 때문에 이름이 겹치면 덮어쓰기 문제가 생깁니다.

따라서:

- basename이 유일하면 원래 이름 유지
- 중복 basename이면 원천 Training/Validation 데이터 경로 hash를 prefix로 붙임

이 규칙으로 데이터가 늘어나도 안전하게 처리합니다.

In [ ]:
basename_counts = Counter(image_df["image_name"])


def processed_image_name(image_member: str) -> str:
    basename = Path(image_member).name

    if basename_counts[basename] == 1:
        return basename

    short_hash = hashlib.sha1(image_member.encode("utf-8")).hexdigest()[:10]
    return f"{short_hash}__{basename}"

# 25. 실제 processed 생성

각 유지 이미지에 대해 다음 작업을 합니다.

1. 원본 JPG bytes 복사
2. 해당 JSON을 다시 읽음
3. 학습 대상 클래스 객체만 선택
4. BOX/POLYGON을 xyxy bbox로 변환
5. 실제 이미지 크기로 clip
6. YOLO normalized 좌표로 변환
7. `class_id xc yc w h` txt 저장

변환 과정 자체도 다시 보고서로 남깁니다.

In [ ]:
processed_image_rows = []
processed_object_rows = []
preprocess_error_rows = []

keep_lookup = image_df.set_index("json_member")["keep_for_processed"].to_dict()
image_info_lookup = image_df.set_index("json_member").to_dict("index")

for json_member in sorted(json_members):
    if not keep_lookup.get(json_member, False):
        continue

    info = image_info_lookup[json_member]

    source_split = info["split"]
    output_split = "train" if source_split == "Training" else "val"

    image_path = Path(info["image_member"])
    json_path = Path(json_member)

    output_name = processed_image_name(str(image_path))
    output_image_path = OUTPUT_DATASET_DIR / "images" / output_split / output_name
    output_label_path = (
        OUTPUT_DATASET_DIR / "labels" / output_split / f"{Path(output_name).stem}.txt"
    )

    # 원본 bytes를 그대로 복사합니다. 여기서 리사이즈/JPEG 재압축은 하지 않습니다.
    shutil.copy2(image_path, output_image_path)

    try:
        with open(json_path, "r", encoding="utf-8-sig") as file:
            payload = json.load(file)
    except UnicodeDecodeError:
        with open(json_path, "r", encoding="cp949") as file:
            payload = json.load(file)

    width = int(info["width"])
    height = int(info["height"])

    label_lines = []
    kept_object_count = 0

    for object_index, bounding in enumerate(payload.get("Bounding") or []):
        major = canonical_major(bounding.get("CLASS", "UNKNOWN"))
        detail = str(bounding.get("DETAILS", "UNKNOWN")).strip()
        class_name = f"{major}/{detail}"

        if class_name not in CLASS_TO_ID:
            continue

        try:
            x1, y1, x2, y2, geometry = bounding_to_xyxy(bounding)

            x1 = float(np.clip(x1, 0, width))
            x2 = float(np.clip(x2, 0, width))
            y1 = float(np.clip(y1, 0, height))
            y2 = float(np.clip(y2, 0, height))

            if x2 <= x1 or y2 <= y1:
                raise ValueError("clip 후 bbox가 유효하지 않습니다.")

            xc, yc, bw, bh = xyxy_to_yolo(
                x1, y1, x2, y2,
                image_width=width,
                image_height=height,
            )

            normalized = np.array([xc, yc, bw, bh], dtype=float)

            if not np.all((normalized >= 0) & (normalized <= 1)):
                raise ValueError(f"normalized bbox 범위 오류: {normalized.tolist()}")

            if bw <= 0 or bh <= 0:
                raise ValueError("normalized width/height가 0 이하")

            class_id = CLASS_TO_ID[class_name]

            label_lines.append(
                f"{class_id} {xc:.8f} {yc:.8f} {bw:.8f} {bh:.8f}"
            )

            processed_object_rows.append({
                "split": output_split,
                "processed_image": output_name,
                "processed_label": output_label_path.name,
                "source_image": str(image_path),
                "source_json": str(json_path),
                "object_index": object_index,
                "class_id": class_id,
                "class_name": class_name,
                "source_geometry": geometry,
                "x_center": xc,
                "y_center": yc,
                "width": bw,
                "height": bh,
            })

            kept_object_count += 1

        except Exception as error:
            preprocess_error_rows.append({
                "split": output_split,
                "source_image": str(image_path),
                "source_json": str(json_path),
                "object_index": object_index,
                "error": repr(error),
            })

    if kept_object_count == 0:
        if output_image_path.exists():
            output_image_path.unlink()

        preprocess_error_rows.append({
            "split": output_split,
            "source_image": str(image_path),
            "source_json": str(json_path),
            "object_index": None,
            "error": "Training class 체계에 포함되는 bbox가 없어 이미지 제외",
        })
        continue

    output_label_path.write_text(
        "\n".join(label_lines) + "\n",
        encoding="utf-8",
    )

    processed_image_rows.append({
        "split": output_split,
        "source_split": source_split,
        "source_image": str(image_path),
        "source_json": str(json_path),
        "processed_image": output_name,
        "processed_label": output_label_path.name,
        "objects": kept_object_count,
        "source_condition": info.get("source_condition"),
        "damage_level": info.get("damage_level"),
    })

processed_image_df = pd.DataFrame(processed_image_rows)
processed_object_df = pd.DataFrame(processed_object_rows)
preprocess_error_df = pd.DataFrame(preprocess_error_rows)

print("processed images :", len(processed_image_df))
print("processed objects:", len(processed_object_df))
print("preprocess errors:", len(preprocess_error_df))

# 26. `data.yaml` 생성

YOLO는 `data.yaml`을 통해 다음을 알게 됩니다.

- train 이미지 위치
- validation 이미지 위치
- 클래스 id와 이름

`path`는 현재 processed 절대경로로 기록합니다.
두 번째 노트북에서는 데이터 폴더가 이동했을 가능성까지 고려해 현재 경로로 다시 보정합니다.

In [ ]:
data_yaml = {
    "path": str(OUTPUT_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {
        class_id: class_name
        for class_id, class_name in ID_TO_CLASS.items()
    },
}

DATA_YAML_PATH = OUTPUT_DATASET_DIR / "data.yaml"

with open(DATA_YAML_PATH, "w", encoding="utf-8") as file:
    yaml.safe_dump(
        data_yaml,
        file,
        allow_unicode=True,
        sort_keys=False,
    )

print(DATA_YAML_PATH.read_text(encoding="utf-8")[:5000])

# 27. processed용 보고서 CSV 저장

이 보고서들은 “어떤 데이터로 모델을 학습했는가?”를 나중에 다시 추적할 수 있게 해줍니다.

특히 팀 프로젝트에서는 모델 성능 숫자만 남기기보다 **학습 데이터 버전과 제외 사유를 함께 기록**하는 것이 중요합니다.

In [ ]:
class_mapping_df = pd.DataFrame({
    "class_id": range(len(CLASS_NAMES)),
    "class_name": CLASS_NAMES,
})

processed_support_df = (
    processed_object_df
    .groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["train", "val"], fill_value=0)
    .reindex(CLASS_NAMES, fill_value=0)
)

processed_support_df["total"] = processed_support_df["train"] + processed_support_df["val"]
processed_support_df = processed_support_df.reset_index()

image_df.to_csv(REPORT_DIR / "raw_image_audit.csv", index=False, encoding="utf-8-sig")
object_df.to_csv(REPORT_DIR / "raw_object_audit.csv", index=False, encoding="utf-8-sig")
issue_df.to_csv(REPORT_DIR / "raw_issues.csv", index=False, encoding="utf-8-sig")
excluded_df.to_csv(REPORT_DIR / "excluded_images.csv", index=False, encoding="utf-8-sig")
processed_image_df.to_csv(REPORT_DIR / "processed_images.csv", index=False, encoding="utf-8-sig")
processed_object_df.to_csv(REPORT_DIR / "processed_objects.csv", index=False, encoding="utf-8-sig")
preprocess_error_df.to_csv(REPORT_DIR / "preprocess_errors.csv", index=False, encoding="utf-8-sig")
class_mapping_df.to_csv(REPORT_DIR / "class_mapping.csv", index=False, encoding="utf-8-sig")
processed_support_df.to_csv(REPORT_DIR / "class_support_processed.csv", index=False, encoding="utf-8-sig")

sampling_plan_df.to_csv(REPORT_DIR / "class_sampling_plan.csv", index=False, encoding="utf-8-sig")

selected_manifest_df = image_df[image_df["selected_for_processed"]].copy()
selected_manifest_df.to_csv(REPORT_DIR / "selected_dataset_manifest.csv", index=False, encoding="utf-8-sig")

dataset_metadata = {
    "dataset_variant": DATASET_VARIANT,
    "use_damaged_data": bool(USE_DAMAGED_DATA),
    "damaged_ratio_target": float(DAMAGED_RATIO),
    "max_train_images_per_class": int(MAX_TRAIN_IMAGES_PER_CLASS),
    "sampling_seed": int(SAMPLING_SEED),
    "output_dataset_dir": str(OUTPUT_DATASET_DIR),
}

with open(REPORT_DIR / "dataset_metadata.json", "w", encoding="utf-8") as file:
    json.dump(dataset_metadata, file, ensure_ascii=False, indent=2)

print("보고서 저장 위치:", REPORT_DIR)
for path in sorted(REPORT_DIR.glob("*.csv")):
    print("-", path.name)

# 28. processed 최종 규모 확인

원본 수와 processed 수가 다른 이유를 반드시 확인합니다.

현재 정책에서는 폴더 목표 라벨이 JSON에 없는 의심 샘플이 제외되므로 processed가 원본보다 조금 작아지는 것이 정상입니다.

In [ ]:
processed_summary = (
    processed_image_df
    .groupby("split")
    .agg(
        images=("processed_image", "count"),
        objects=("objects", "sum"),
        unique_target_classes=("folder_target_class", "nunique"),
    )
    .reindex(["train", "val"])
)

display(processed_summary)

display(processed_support_df)

print("Validation 객체가 0개인 클래스")
display(
    processed_support_df[
        processed_support_df["val"].eq(0)
    ][["class_name", "train", "val"]]
)

# 29. processed 파일 1:1 대응 검사

모든 processed 이미지에는 같은 stem의 label txt가 있어야 합니다.

예:

```text
images/train/A.jpg
labels/train/A.txt
```

이 단계에서 하나라도 빠져 있으면 학습 전에 중단합니다.

In [ ]:
file_pair_errors = []

for split in ["train", "val"]:
    image_paths = sorted((OUTPUT_DATASET_DIR / "images" / split).glob("*"))
    label_paths = sorted((OUTPUT_DATASET_DIR / "labels" / split).glob("*.txt"))

    image_stems = {path.stem for path in image_paths if path.is_file()}
    label_stems = {path.stem for path in label_paths if path.is_file()}

    missing_labels = sorted(image_stems - label_stems)
    missing_images = sorted(label_stems - image_stems)

    for stem in missing_labels:
        file_pair_errors.append((split, stem, "missing_label"))

    for stem in missing_images:
        file_pair_errors.append((split, stem, "missing_image"))

    print(
        split,
        "images=", len(image_stems),
        "labels=", len(label_stems),
        "missing_labels=", len(missing_labels),
        "missing_images=", len(missing_images),
    )

if file_pair_errors:
    display(pd.DataFrame(file_pair_errors, columns=["split", "stem", "error"]))
    raise ValueError("processed 이미지-label 1:1 대응 오류가 있습니다.")

print("Image-label pairing: PASSED")

# 30. YOLO txt를 다시 읽어서 범위 검사

전처리 코드가 txt를 만들었다고 끝이 아닙니다.
최종 저장된 파일을 다시 읽어 다음을 확인합니다.

- 한 줄이 5개 값인지
- class id가 0~85 범위인지
- 좌표가 모두 0~1인지
- width / height가 0보다 큰지

이것은 “생성 코드”와 “최종 파일” 사이의 마지막 안전검사입니다.

In [ ]:
label_validation_errors = []
label_object_count = 0

for split in ["train", "val"]:
    for label_path in sorted((OUTPUT_DATASET_DIR / "labels" / split).glob("*.txt")):
        text = label_path.read_text(encoding="utf-8").strip()

        if not text:
            label_validation_errors.append({
                "split": split,
                "label": str(label_path),
                "error": "empty_label",
            })
            continue

        for line_number, line in enumerate(text.splitlines(), 1):
            try:
                parts = line.split()

                if len(parts) != 5:
                    raise ValueError(f"5개 값이 아니라 {len(parts)}개 값")

                class_value, xc, yc, bw, bh = map(float, parts)
                class_id = int(class_value)

                if class_value != class_id:
                    raise ValueError("class id가 정수가 아님")

                if class_id not in ID_TO_CLASS:
                    raise ValueError(f"알 수 없는 class id={class_id}")

                coords = np.array([xc, yc, bw, bh], dtype=float)

                if not np.all((coords >= 0) & (coords <= 1)):
                    raise ValueError(f"0~1 범위를 벗어난 좌표: {coords.tolist()}")

                if bw <= 0 or bh <= 0:
                    raise ValueError("bbox width/height가 0 이하")

                label_object_count += 1

            except Exception as error:
                label_validation_errors.append({
                    "split": split,
                    "label": str(label_path),
                    "line": line_number,
                    "content": line,
                    "error": repr(error),
                })

print("YOLO txt에서 다시 읽은 객체 수:", label_object_count)
print("라벨 검증 오류 수:", len(label_validation_errors))

if label_validation_errors:
    display(pd.DataFrame(label_validation_errors).head(50))
    raise ValueError("processed YOLO label 검증 오류가 있습니다.")

print("YOLO label validation: PASSED")

# 31. processed Train-Val 누수 다시 검사

원본에서 중복이 없더라도 processed 생성 후 다시 확인합니다.
이번에는 이미 계산한 SHA-1을 사용합니다.

In [ ]:
processed_cross_split_dups = (
    processed_image_df
    .groupby("sha1")
    .filter(lambda group: group["split"].nunique() > 1)
)

print("processed Train-Val exact duplicate rows:", len(processed_cross_split_dups))

if len(processed_cross_split_dups):
    display(processed_cross_split_dups)
    raise ValueError("processed Train-Val 데이터 누수가 있습니다.")

print("Processed leakage check: PASSED")

# 32. YOLO txt bbox를 다시 픽셀 좌표로 복원하는 함수

마지막으로 실제 이미지 위에 bbox를 그려 봅니다.

YOLO txt는 정규화 좌표이므로 시각화를 위해 다시 pixel 좌표로 복원해야 합니다.

In [ ]:
def read_yolo_label_as_xyxy(label_path: Path, image_width: int, image_height: int):
    classes = []
    boxes = []

    text = label_path.read_text(encoding="utf-8").strip()

    for line in text.splitlines():
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)

        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height

        classes.append(class_id)
        boxes.append([x1, y1, x2, y2])

    return np.asarray(classes, dtype=int), np.asarray(boxes, dtype=np.float32).reshape(-1, 4)


def draw_boxes(image_bgr, classes, boxes):
    output = image_bgr.copy()

    for class_id, box in zip(classes, boxes):
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # OpenCV 기본 폰트는 한글 지원이 제한적이라 이미지에는 class id만 표시합니다.
        cv2.putText(
            output,
            str(class_id),
            (x1, max(20, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
            cv2.LINE_AA,
        )

    return output

# 33. processed bbox 샘플 눈으로 확인

Train / Validation 각각 여러 장을 랜덤하게 골라 실제 bbox를 확인합니다.

확인 포인트:

- bbox가 실제 객체를 감싸는가?
- 완전히 엉뚱한 곳에 그려지지 않았는가?
- 너무 크거나 너무 작게 변환되지 않았는가?
- class id와 표의 class name이 맞는가?

In [ ]:
NUM_VISUAL_SAMPLES_PER_SPLIT = 4
rng = np.random.default_rng(SEED)

for split in ["train", "val"]:
    split_rows = processed_image_df[processed_image_df["split"].eq(split)]
    sample_count = min(NUM_VISUAL_SAMPLES_PER_SPLIT, len(split_rows))
    sample_indices = rng.choice(split_rows.index.to_numpy(), size=sample_count, replace=False)

    print("=" * 90)
    print(split.upper())

    for index in sample_indices:
        row = processed_image_df.loc[index]

        image_path = OUTPUT_DATASET_DIR / "images" / split / row["processed_image"]
        label_path = OUTPUT_DATASET_DIR / "labels" / split / row["processed_label"]

        image = cv2.imread(str(image_path))
        height, width = image.shape[:2]
        classes, boxes = read_yolo_label_as_xyxy(label_path, width, height)
        drawn = draw_boxes(image, classes, boxes)

        plt.figure(figsize=(10, 7))
        plt.imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
        plt.title(f"{split}: {row['processed_image']}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

        display(pd.DataFrame({
            "class_id": classes,
            "class_name": [ID_TO_CLASS[int(class_id)] for class_id in classes],
        }))

# 34. processed 클래스 분포 시각화

이 그래프는 **최종 모델이 실제로 학습하게 될 객체 수**입니다.

특히 Validation 객체가 0인 클래스는 해당 Validation set만으로는 그 클래스 AP를 제대로 평가할 수 없습니다.
두 번째 노트북에서는 전체 mAP와 함께 클래스별 support를 항상 같이 표시합니다.

In [ ]:
plot_df = processed_support_df.copy()

plt.figure(figsize=(16, 6))
plt.bar(np.arange(len(plot_df)), plot_df["train"])
plt.xlabel("Class index")
plt.ylabel("Train objects")
plt.title("Processed Train Object Count per Class")
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
plt.bar(np.arange(len(plot_df)), plot_df["val"])
plt.xlabel("Class index")
plt.ylabel("Validation objects")
plt.title("Processed Validation Object Count per Class")
plt.tight_layout()
plt.show()

# 35. EDA / 전처리 요약 파일 만들기

사람이 읽기 쉬운 한 줄 요약 CSV도 저장합니다.
이 파일은 두 번째 노트북에서 “어떤 processed를 사용했는지” 확인할 때 사용할 수 있습니다.

In [ ]:
summary_rows = [
    {"key": "dataset_variant", "value": DATASET_VARIANT},
    {"key": "use_damaged_data", "value": USE_DAMAGED_DATA},
    {"key": "damaged_ratio_target", "value": DAMAGED_RATIO},
    {"key": "max_train_images_per_class", "value": MAX_TRAIN_IMAGES_PER_CLASS},
    {"key": "sampling_seed", "value": SAMPLING_SEED},
    {"key": "output_dataset_dir", "value": str(OUTPUT_DATASET_DIR)},
    {"key": "source_root", "value": str(PROCESSED_DIR)},
    {"key": "training_source_dir", "value": str(TRAINING_SOURCE_DIR)},
    {"key": "validation_source_dir", "value": str(VALIDATION_SOURCE_DIR)},
    {"key": "raw_images", "value": len(image_df)},
    {"key": "raw_objects", "value": len(object_df)},
    {"key": "training_classes", "value": len(CLASS_NAMES)},
    {"key": "raw_training_images", "value": int((image_df["split"] == "Training").sum())},
    {"key": "raw_validation_images", "value": int((image_df["split"] == "Validation").sum())},
    {"key": "excluded_images", "value": len(excluded_df)},
    {"key": "processed_train_images", "value": int((processed_image_df["split"] == "train").sum())},
    {"key": "processed_val_images", "value": int((processed_image_df["split"] == "val").sum())},
    {"key": "processed_train_objects", "value": int((processed_object_df["split"] == "train").sum())},
    {"key": "processed_val_objects", "value": int((processed_object_df["split"] == "val").sum())},
    {"key": "polygon_to_bbox_objects", "value": int((processed_object_df["source_geometry"] == "POLYGON->BOX").sum())},
    {"key": "preprocess_errors", "value": len(preprocess_error_df)},
    {"key": "val_classes_without_objects", "value": int((processed_support_df["val"] == 0).sum())},
]

preprocess_summary_df = pd.DataFrame(summary_rows)
preprocess_summary_df.to_csv(
    REPORT_DIR / "preprocess_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(preprocess_summary_df)

# 36. 최종 폴더 트리 확인

이 셀에서 보이는 구조가 두 번째 노트북의 입력이 됩니다.

In [ ]:
def print_tree(root: Path, max_depth=3):
    root = root.resolve()
    print(root.name + "/")

    for path in sorted(root.rglob("*")):
        relative = path.relative_to(root)

        if len(relative.parts) > max_depth:
            continue

        indent = "    " * (len(relative.parts) - 1)
        suffix = "/" if path.is_dir() else ""
        print(f"{indent}├── {path.name}{suffix}")


print_tree(OUTPUT_DATASET_DIR, max_depth=3)

# 37. 전처리 결과 해석과 비교 실험 방법

## 이 노트북에서 통제한 것

### 1) 총 학습량을 가능한 한 동일하게 유지

`USE_DAMAGED_DATA=True`에서 damaged를 단순 추가하지 않고 clean 일부와 교체합니다.
그래서 클래스별 총 학습 이미지 수가 clean-only 기준과 같습니다.

### 2) 같은 clean baseline 표본 사용

두 모드가 완전히 다른 clean 표본을 랜덤하게 뽑지 않도록 deterministic seed를 사용합니다.
with-damaged의 clean 데이터는 clean-only baseline 표본의 부분집합입니다.

### 3) Validation 동일

Validation은 두 모드에서 동일합니다.
따라서 Validation 성능 차이가 train 구성 차이와 더 직접적으로 연결됩니다.

### 4) 파손 부족 클래스는 중복 복제하지 않음

목표 damaged가 부족한 클래스는 가능한 damaged만 사용하고 나머지는 clean으로 채웁니다.
실제 비율은 `class_sampling_plan.csv`에서 확인할 수 있습니다.

---

## 가장 공정한 성능 비교 방법

파손 데이터 효과 자체를 보려면 다음 조건을 모두 동일하게 유지하세요.

- model size
- pretrained weight
- epoch
- imgsz
- batch
- optimizer / learning rate
- augmentation 정책
- seed
- Validation

그리고 **Training 데이터 구성만** 바꿉니다.

```text
실험 A: clean_only
실험 B: with_damaged_20pct
```

이렇게 해야 "파손 데이터 포함"이라는 한 변수의 효과를 가장 깨끗하게 비교할 수 있습니다.

하이퍼파라미터나 augmentation까지 각 실험에서 따로 다시 최적화하면
그 결과는 "각 파이프라인을 최대로 최적화했을 때의 비교"가 되고,
파손 데이터 자체의 순수 효과를 보는 ablation과는 의미가 달라집니다.

# 38. 전처리 완료 체크리스트

아래를 확인하고 Kaggle 학습으로 넘어가세요.

- [ ] `USE_DAMAGED_DATA`가 의도한 값인지 확인했다.
- [ ] `DAMAGED_RATIO`가 의도한 비율인지 확인했다.
- [ ] `class_sampling_plan.csv`에서 클래스별 clean/damaged 실제 사용량을 확인했다.
- [ ] damaged 부족 클래스의 `damaged_shortage`를 확인했다.
- [ ] clean-only와 damaged-mix의 `selected_total`이 같은지 비교했다.
- [ ] Validation 데이터는 두 모드에서 동일하다.
- [ ] processed 이미지와 label이 1:1 대응한다.
- [ ] 모든 YOLO bbox 좌표가 0~1 범위다.
- [ ] class id가 `data.yaml`과 일치한다.
- [ ] 실제 bbox 시각화를 확인했다.

현재 결과 폴더는 다음 둘 중 하나입니다.

```text
../../data/processed/model_ready/clean_only/
../../data/processed/model_ready/with_damaged_20pct/
```

### 권장 실행 순서

1. `USE_DAMAGED_DATA=False` 실행
2. `clean_only` 폴더 보존
3. `USE_DAMAGED_DATA=True` 실행
4. `with_damaged_20pct` 폴더 보존
5. 두 dataset을 Kaggle에 업로드
6. 동일한 학습 설정으로 성능 비교